# Prepare data for natural experiments
1. Bilboard Hot 100
2. Lottery Mega Millions
3. Market Data (S&P)
4. NBA Games

In [ ]:
import os
from pathlib import Path

def find_project_root(marker=".git"):
    path = os.getcwd()
    while path != os.path.dirname(path):
        if marker in os.listdir(path):
            return path
        path = os.path.dirname(path)
    return None

root = find_project_root()
if root is not None:
    os.chdir(root)

import pandas as pd

DATA_FOLDER = "data/naturalistic_data"
os.makedirs(DATA_FOLDER, exist_ok=True)

## Bilboard hot 100 data

Data is hosted on:
https://github.com/mhollingshead/billboard-hot-100

In [ ]:
import requests
# https://github.com/mhollingshead/billboard-hot-100
response = requests.get("https://raw.githubusercontent.com/mhollingshead/billboard-hot-100/main/all.json")
data = response.json()

In [ ]:
df = []
for week in data:
    entries = []
    date = week['date']
    for entry in week['data']:
        rank_last_week =  entry['last_week']
        if rank_last_week is None:
            rank_last_week = 0
        else:
            rank_last_week = int(rank_last_week)
        entries.append({
            'date': date,
            'song': entry['song'],
            'artist': entry['artist'],
            'rank': entry['this_week'],
            'rank_last_week': rank_last_week,           
            'peak_rank': entry['peak_position'],
            'weeks_on_chart': entry['weeks_on_chart']
        })
    df.extend(entries)
df = pd.DataFrame(df)

df['date'] = pd.to_datetime(df['date'])
df = df[df['date'] >= '2000-01-01'].copy()
print(df.shape)
df.head()

In [ ]:
df.to_csv(f'{DATA_FOLDER}/billboard_100_2000-2025.csv', index=False)

## Lottery Data

Source data can be downloaded from:
https://catalog.data.gov/dataset/lottery-mega-millions-winning-numbers-beginning-2002


In [ ]:
df = pd.read_csv('~/Desktop/Lottery_Mega_Millions_Winning_Numbers__Beginning_2002.csv')
df.head()

In [ ]:
df.rename(columns={'Draw Date': 'date', 'Winning Numbers': 'winning_numbers', 'Mega Ball': 'mega_ball', 'Multiplier': 'multiplier'}, inplace=True)
df['date'] = pd.to_datetime(df['date'])
df.head()

df.to_csv(f'{DATA_FOLDER}/mega_millions_processed.csv', index=False)

## Market Data

Navigate to https://wrds-jupyter.wharton.upenn.edu/, and execute the following code in the jupyter hub:

```
import wrds
import pandas as pd

db = wrds.Connection()

sql_query = """
    SELECT date, spindx, sprtrn 
    FROM crsp.dsi 
    WHERE date >= '2002-01-01' AND date <= '2024-12-31'
"""
sp500_data = db.raw_sql(sql_query)
sp500_data['date'] = pd.to_datetime(sp500_data['date'])
sp500_data.rename(columns={'spindx': 'close'}, inplace=True)
sp500_data.to_csv("crsp_sp500_daily_2002_2024.csv", index=False)
print("Saved!")
```

In [ ]:
df['date'] = pd.to_datetime(df['date'])
df.sort_values(by=['date'], inplace=True)

df['ticker'] = "s&p"
os.makedirs(f'{DATA_FOLDER}/market_data', exist_ok=True)
df.to_csv(f'{DATA_FOLDER}/market_data/snp.csv', index=False)

## NBA Data

In [ ]:
import pandas as pd
import json
import time
from tqdm import tqdm
from nba_api.stats.endpoints import leaguegamefinder
from nba_api.stats.endpoints import boxscoretraditionalv3
from nba_api.stats.library.parameters import Season

# --- Configuration ---
# Set the range of NBA seasons you want to collect data for.
START_SEASON = 2000
END_SEASON = 2025 

# Output filenames
SUMMARY_CSV_FILE = 'nba_game_summaries.csv'
BOX_SCORE_JSON_FILE = 'nba_player_box_scores.json'

# Delay to prevent hitting API rate limits (in seconds)
API_DELAY_SECONDS = 0.6 

# --- Column Name Mapping ---
# Map the camelCase names from boxscoretraditionalv3 (V3) to the final
# SNAKE_CASE names for consistency in the final CSV output.
V3_TO_CSV_MAP = {
    'points': 'PTS',
    'reboundsTotal': 'REB',
    'assists': 'AST',
    'steals': 'STL',
    'blocks': 'BLK',
    # Note: 'teamName' is used for matching but stored as TEAM_NAME
}

def get_season_years(start_year, end_year):
    """Generates a list of season strings (e.g., '2021-22') for the given range."""
    seasons = []
    for year in range(start_year, end_year):
        seasons.append(f"{year}-{str(year + 1)[2:]}")
    return seasons

def get_nba_data(season_years, get_box_scores=True):
    """
    Fetches NBA game data and detailed player box scores for the specified seasons.
    
    Returns:
        tuple: (pd.DataFrame of game summaries, dict of player box scores)
    """
    all_game_summaries = []
    all_player_box_scores = {}
    
    print(f"Starting data collection for seasons: {', '.join(season_years)}")

    for season in tqdm(season_years):
        print(f"\n--- Processing Season: {season} ---")
        
        # 1. Fetch all games for the season (uses legacy SNAKE_CASE columns)
        try:
            gamefinder = leaguegamefinder.LeagueGameFinder(
                season_nullable=season,
                league_id_nullable='00' 
            )
            # This uses the older API format with SNAKE_CASE
            games_df = gamefinder.get_data_frames()[0]
        except Exception as e:
            print(f"Error fetching game list for {season}: {e}. Skipping season.")
            continue

        # --- Convert legacy SNAKE_CASE columns to camelCase for consistency ---
        games_df.rename(columns={
            'GAME_ID': 'gameId',
            'MATCHUP': 'matchup',
            'TEAM_NAME': 'teamName',
            'GAME_DATE': 'gameDate',
            # **CRITICAL FIX: Rename TEAM_ID for consistent matching**
            'TEAM_ID': 'teamId' 
        }, inplace=True)
        # Ensure teamId is integer for matching
        try:
            games_df['teamId'] = games_df['teamId'].astype(int)
        except Exception as e:
            pass
        # ---------------------------------------------------------------------

        unique_games = games_df.drop_duplicates(subset=['gameId'])
        game_ids = unique_games['gameId'].tolist()
        
        print(f"Found {len(game_ids)} unique games in {season}. Starting box score collection.")
        
        if not get_box_scores:
            all_game_summaries.append(games_df)
            continue
        
        # 2. Iterate through unique games to get detailed player stats and team outcomes
        for game_id in tqdm(game_ids):

            try:
                # Fetch detailed box score for the game
                # This returns DataFrames with camelCase columns (V3 endpoint)
                boxscore = boxscoretraditionalv3.BoxScoreTraditionalV3(game_id=game_id)
                
                # Get the player stats (DataFrame [0]) 
                player_stats_df = boxscore.get_data_frames()[0]
                
                # Get the team stats (DataFrame [1]) 
                team_stats_df = boxscore.get_data_frames()[1]
                
                # Ensure teamId is integer in V3 data for matching
                team_stats_df['teamId'] = team_stats_df['teamId'].astype(int)

                # --- Player Box Score Structuring (JSON requirement) ---
                player_data = player_stats_df.to_dict('records')
                all_player_box_scores[game_id] = player_data
                
                # --- Game Summary Structuring (CSV requirement) ---
                # 1. Get the two rows corresponding to the game from the full list (now camelCase)
                game_rows = games_df[games_df['gameId'] == game_id]
                
                # 2. Determine Home and Away Team based on matchup
                home_row = game_rows[game_rows['matchup'].str.contains('vs.')]
                if home_row.empty: home_row = game_rows.iloc[[0]] 
                home_row = home_row.iloc[0]

                away_row = game_rows[game_rows['matchup'].str.contains('@')]
                if away_row.empty: away_row = game_rows.iloc[[1]] 
                away_row = away_row.iloc[0]
                
                # Get the unique numerical IDs
                home_team_id_finder = home_row['teamId']
                away_team_id_finder = away_row['teamId']
                game_date = home_row['gameDate'] 

                # 3. Find the corresponding team stats in the box score data using the unique TEAM ID
                
                # Filter using the robust 'teamId' column
                home_team_filter = team_stats_df[team_stats_df['teamId'] == home_team_id_finder]
                away_team_filter = team_stats_df[team_stats_df['teamId'] == away_team_id_finder]

                if home_team_filter.empty or away_team_filter.empty:
                     # Fallback error reporting in case the ID itself is wrong (unlikely)
                     raise ValueError(f"Could not find corresponding team stats in Box Score data for IDs {home_team_id_finder} or {away_team_id_finder} using V3 'teamId' columns.")

                home_team_row = home_team_filter.iloc[0]
                away_team_row = away_team_filter.iloc[0]
                
                # 4. Create a single summary dictionary for the CSV 
                summary_data = {
                    'GAME_ID': game_id,
                    'GAME_DATE': game_date,
                    'HOME_TEAM': home_team_row['teamName'], # Use V3 name
                    'AWAY_TEAM': away_team_row['teamName'], # Use V3 name
                }
                
                # Collect required outcomes using the V3 camelCase keys, mapping to CSV SNAKE_CASE
                for v3_key, csv_key in V3_TO_CSV_MAP.items():
                    # Access data using the correct camelCase key (e.g., 'points')
                    if v3_key not in home_team_row:
                        raise KeyError(f"Missing expected V3 column '{v3_key}' in box score.")
                        
                    summary_data[f'HOME_{csv_key}'] = home_team_row[v3_key]
                    summary_data[f'AWAY_{csv_key}'] = away_team_row[v3_key]
                    
                all_game_summaries.append(summary_data)
                
            except Exception as e:
                # Log the specific error and game ID
                print(f"  Warning: Could not fetch box score for Game ID {game_id}. Error: {e}")
                
            time.sleep(API_DELAY_SECONDS)

    # 3. Consolidate and return
    if get_box_scores:
        summary_df = pd.DataFrame(all_game_summaries)
    else:
        summary_df = pd.concat(all_game_summaries, ignore_index=True)
        
    return summary_df, all_player_box_scores

def save_data(summary_df, box_scores_dict):
    """Saves the collected data to CSV and JSON files."""
    
    # Save the summary dataframe to CSV
    summary_df.to_csv(f'{DATA_FOLDER}/{SUMMARY_CSV_FILE}', index=False)
    print(f"\nSuccessfully saved game summaries to: {DATA_FOLDER}/{SUMMARY_CSV_FILE}")
    print(f"Total summary rows: {len(summary_df)}")
    
    # Save the player box scores dictionary to JSON
    with open(f'{DATA_FOLDER}/{BOX_SCORE_JSON_FILE}', 'w') as f:
        json.dump(box_scores_dict, f, indent=4)
    print(f"Successfully saved detailed player box scores to: {DATA_FOLDER}/{BOX_SCORE_JSON_FILE}")
    print(f"Total games with detailed box scores: {len(box_scores_dict)}")


if __name__ == "__main__":
    
    seasons_to_process = get_season_years(START_SEASON, END_SEASON)
    
    if not seasons_to_process:
        print("Error: START_SEASON must be less than END_SEASON.")
    else:
        summary_df, box_scores_dict = get_nba_data(seasons_to_process, get_box_scores=False)

In [ ]:
print(len(summary_df))
summary_df.head()

In [ ]:
summary_df.to_csv(f'{DATA_FOLDER}/nba_data_{START_SEASON}-{END_SEASON}.csv', index=False)